<a href="https://colab.research.google.com/github/virodulacharishma0506-del/GENAI/blob/main/Week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-dotenv

In [ ]:
%%writefile .env

INPUT_PRICE_PER_MILLION=0.30
OUTPUT_PRICE_PER_MILLION=1.20

REQUESTS_PER_DAY=2000
AVG_INPUT_TOKENS=1200
AVG_OUTPUT_TOKENS=300
PEAK_REQUESTS_PER_MINUTE=30

RPM_LIMIT=60
TPM_LIMIT=100000

MONTHLY_BUDGET_USD=50
DAILY_BUDGET_USD=2
SAFETY_MARGIN_PERCENT=20

Writing .env


In [ ]:
%%writefile budget.py

import os
from dataclasses import dataclass
from dotenv import load_dotenv

@dataclass
class FeatureBudget:
    requests_per_day: int
    avg_input_tokens: int
    avg_output_tokens: int
    peak_requests_per_minute: int
    input_price_per_million: float
    output_price_per_million: float
    rpm_limit: int
    tpm_limit: int
    monthly_budget_usd: float
    daily_budget_usd: float
    safety_margin_percent: float

    @classmethod
    def from_env(cls):
        load_dotenv()

        return cls(
            requests_per_day=int(os.getenv("REQUESTS_PER_DAY")),
            avg_input_tokens=int(os.getenv("AVG_INPUT_TOKENS")),
            avg_output_tokens=int(os.getenv("AVG_OUTPUT_TOKENS")),
            peak_requests_per_minute=int(os.getenv("PEAK_REQUESTS_PER_MINUTE")),
            input_price_per_million=float(os.getenv("INPUT_PRICE_PER_MILLION")),
            output_price_per_million=float(os.getenv("OUTPUT_PRICE_PER_MILLION")),
            rpm_limit=int(os.getenv("RPM_LIMIT")),
            tpm_limit=int(os.getenv("TPM_LIMIT")),
            monthly_budget_usd=float(os.getenv("MONTHLY_BUDGET_USD")),
            daily_budget_usd=float(os.getenv("DAILY_BUDGET_USD")),
            safety_margin_percent=float(os.getenv("SAFETY_MARGIN_PERCENT"))
        )

    def calculate(self):
        if min(
            self.requests_per_day,
            self.avg_input_tokens,
            self.avg_output_tokens,
            self.peak_requests_per_minute
        ) < 0:
            raise ValueError("Usage values cannot be negative")

        if self.safety_margin_percent < 0:
            raise ValueError("Safety margin cannot be negative")

        input_tokens = self.requests_per_day * self.avg_input_tokens
        output_tokens = self.requests_per_day * self.avg_output_tokens
        total_tokens = input_tokens + output_tokens

        daily_input_cost = (
            input_tokens / 1_000_000
        ) * self.input_price_per_million

        daily_output_cost = (
            output_tokens / 1_000_000
        ) * self.output_price_per_million

        daily_cost = daily_input_cost + daily_output_cost
        monthly_cost = daily_cost * 30

        margin = 1 + self.safety_margin_percent / 100

        peak_rpm = self.peak_requests_per_minute * margin
        peak_tpm = (
            self.peak_requests_per_minute
            * (self.avg_input_tokens + self.avg_output_tokens)
            * margin
        )

        return {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "daily_cost": daily_cost,
            "monthly_cost": monthly_cost,
            "peak_rpm": peak_rpm,
            "peak_tpm": peak_tpm,
            "rpm_ok": peak_rpm <= self.rpm_limit,
            "tpm_ok": peak_tpm <= self.tpm_limit,
            "daily_budget_ok": daily_cost <= self.daily_budget_usd,
            "monthly_budget_ok": monthly_cost <= self.monthly_budget_usd
        }


def main():
    budget = FeatureBudget.from_env()
    result = budget.calculate()

    print("===== TOKEN ECONOMICS REPORT =====")
    print("Input tokens per day:", result["input_tokens"])
    print("Output tokens per day:", result["output_tokens"])
    print("Total tokens per day:", result["total_tokens"])
    print("Daily cost: $", round(result["daily_cost"], 2))
    print("Monthly cost: $", round(result["monthly_cost"], 2))
    print("Peak RPM with margin:", result["peak_rpm"])
    print("Peak TPM with margin:", result["peak_tpm"])

    print("\n===== LIMIT CHECKS =====")
    print("RPM check:", result["rpm_ok"])
    print("TPM check:", result["tpm_ok"])
    print("Daily budget check:", result["daily_budget_ok"])
    print("Monthly budget check:", result["monthly_budget_ok"])

    if all([
        result["rpm_ok"],
        result["tpm_ok"],
        result["daily_budget_ok"],
        result["monthly_budget_ok"]
    ]):
        print("\nRecommendation: GO")
    else:
        print("\nRecommendation: NO-GO")


if __name__ == "__main__":
    main()

Writing budget.py


In [ ]:
!python budget.py

===== TOKEN ECONOMICS REPORT =====
Input tokens per day: 2400000
Output tokens per day: 600000
Total tokens per day: 3000000
Daily cost: $ 1.44
Monthly cost: $ 43.2
Peak RPM with margin: 36.0
Peak TPM with margin: 54000.0

===== LIMIT CHECKS =====
RPM check: True
TPM check: True
Daily budget check: True
Monthly budget check: True

Recommendation: GO


In [ ]:
%%writefile test_budget.py

import unittest
from budget import FeatureBudget


def make_budget(**overrides):
    values = {
        "requests_per_day": 1000,
        "avg_input_tokens": 1000,
        "avg_output_tokens": 500,
        "peak_requests_per_minute": 10,
        "input_price_per_million": 1.0,
        "output_price_per_million": 2.0,
        "rpm_limit": 100,
        "tpm_limit": 100000,
        "monthly_budget_usd": 100,
        "daily_budget_usd": 10,
        "safety_margin_percent": 20
    }

    values.update(overrides)
    return FeatureBudget(**values)


class TestFeatureBudget(unittest.TestCase):

    def test_cost_estimate(self):
        budget = make_budget()
        result = budget.calculate()

        self.assertEqual(result["daily_cost"], 2.0)
        self.assertEqual(result["monthly_cost"], 60.0)

    def test_capacity_passes(self):
        result = make_budget().calculate()

        self.assertTrue(result["rpm_ok"])
        self.assertTrue(result["tpm_ok"])

    def test_rpm_limit_fails(self):
        result = make_budget(
            peak_requests_per_minute=100,
            rpm_limit=50
        ).calculate()

        self.assertFalse(result["rpm_ok"])

    def test_monthly_budget_fails(self):
        result = make_budget(
            monthly_budget_usd=20
        ).calculate()

        self.assertFalse(result["monthly_budget_ok"])


if __name__ == "__main__":
    unittest.main(argv=[""], exit=False)

Writing test_budget.py


In [ ]:
!python test_budget.py

....
----------------------------------------------------------------------
Ran 4 tests in 0.000s

OK


In [ ]:
from budget import FeatureBudget

budget = FeatureBudget.from_env()

# Scenario 1: Original requests
result1 = budget.calculate()
print("Original monthly cost:", result1["monthly_cost"])

# Scenario 2: Double the requests
budget.requests_per_day *= 2
result2 = budget.calculate()

print("After doubling requests:")
print("Daily cost:", round(result2["daily_cost"], 2))
print("Monthly cost:", round(result2["monthly_cost"], 2))
print("RPM check:", result2["rpm_ok"])
print("TPM check:", result2["tpm_ok"])
print("Daily budget check:", result2["daily_budget_ok"])
print("Monthly budget check:", result2["monthly_budget_ok"])

Original monthly cost: 43.199999999999996
After doubling requests:
Daily cost: 2.88
Monthly cost: 86.4
RPM check: True
TPM check: True
Daily budget check: False
Monthly budget check: False
